# Quick, Draw! Pictionary: entrenamiento en Google Colab

Este cuaderno entrena los dos clasificadores del proyecto con GPU:

* **CNN** (`--model cnn`): reconoce el dibujo como **imagen** 64x64 (rasterizado a partir de los trazos).
* **BiLSTM** (`--model bilstm`): reconoce el dibujo como **secuencia de trazos** (dx, dy, fin de trazo).

Flujo: comprobar GPU -> montar Google Drive -> clonar el repositorio -> instalar dependencias ->
preprocesar el dataset de Quick, Draw! -> prueba rápida -> entrenamiento completo (CNN y BiLSTM) ->
evaluación en *test* -> curvas y métricas -> copia de `best.pt` a Drive.

**Antes de empezar:** `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4 o superior)`.

Los checkpoints se guardan en Drive y el entrenamiento usa `--resume auto`, así que si Colab
se desconecta basta con volver a ejecutar las celdas: el entrenamiento continúa desde el último epoch.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Montar Google Drive

Drive se usa para lo que debe **sobrevivir** a la sesión de Colab: la caché de los datos crudos
descargados, los checkpoints (`last.pt`, `best.pt`) y los resultados (historial, curvas, métricas).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/QuickDraw_IA'   # carpeta del proyecto en Drive
import os
for sub in ('raw_cache', 'processed', 'checkpoints', 'results', 'best_models'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)
print(os.listdir(DRIVE_ROOT))

## 3. Clonar el repositorio

Variables configurables:

* `REPO_URL` y `BRANCH`: repositorio y rama con el código.
* `PROJECT_SUBDIR`: subcarpeta del repositorio donde están `train.py`, `preprocess.py`, etc.
  (cadena vacía `''` si están en la raíz del repositorio).

Si el repositorio ya está clonado, se actualiza con `git pull`.

In [ ]:
REPO_URL = 'https://github.com/naydegarciatorres-source/PIC_trabajo'
BRANCH = 'feature/ai-classifier'
PROJECT_SUBDIR = ''          # p. ej. 'QuickDraw_IA' si el código está en una subcarpeta
CLONE_DIR = '/content/PIC_trabajo'

import os, subprocess
if os.path.isdir(os.path.join(CLONE_DIR, '.git')):
    subprocess.run(['git', '-C', CLONE_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, CLONE_DIR], check=True)

PROJECT_DIR = os.path.join(CLONE_DIR, PROJECT_SUBDIR) if PROJECT_SUBDIR else CLONE_DIR
assert os.path.isfile(os.path.join(PROJECT_DIR, 'train.py')), f'train.py no encontrado en {PROJECT_DIR}: revisa PROJECT_SUBDIR'
os.chdir(PROJECT_DIR)
print('Directorio de trabajo:', os.getcwd())
print(sorted(os.listdir('.')))

## 4. Instalar dependencias

Colab ya trae `torch` con CUDA; `requirements.txt` solo fija versiones mínimas, por lo que no
reinstala la versión de CUDA de Colab.

In [ ]:
!pip install -q -r requirements.txt
import torch, numpy, cv2, matplotlib
print(torch.__version__, numpy.__version__, cv2.__version__, matplotlib.__version__)

## 5. Comprobar las categorías

`classes.json` define las clases (actualmente 45; la etiqueta es el índice en la lista).
Este comando verifica que todos los nombres existen en la lista oficial de 345 categorías de Quick, Draw!.

In [ ]:
!python preprocess.py --categories-check

## 6. Preprocesado (20 000 dibujos por clase)

`preprocess.py` descarga los ficheros *simplified ndjson* oficiales, filtra `recognized == true`,
elimina duplicados, reparte train/val/test (80/10/10) de forma determinista a partir del `key_id`
(sin fugas entre particiones) y guarda arrays `.npy` compactos.

**¿Dónde guardar los datos?**

| Opción | Ventajas | Inconvenientes |
|---|---|---|
| `/content/...` (disco local de Colab) | Lectura muy rápida durante el entrenamiento (`mmap`) | Se borra al cerrar la sesión |
| Drive (`/content/drive/...`) | Persiste entre sesiones | Lectura más lenta y con latencia; con `mmap` puede frenar el entrenamiento |

Estrategia usada aquí:

1. Los **datos crudos descargados** se cachean en Drive (`raw_cache`), así que volver a ejecutar
   no vuelve a descargar nada.
2. El **dataset procesado** se escribe en el disco local (`/content/data/processed`), que es rápido,
   y se copia a Drive como respaldo. En una sesión nueva, si existe la copia en Drive, se restaura
   en local en lugar de volver a procesar.

Consulta `python preprocess.py --help` para ver todas las opciones disponibles.

In [ ]:
PER_CLASS = 20000
RAW_CACHE = f'{DRIVE_ROOT}/raw_cache'
LOCAL_PROCESSED = '/content/data/processed'
DRIVE_PROCESSED = f'{DRIVE_ROOT}/processed/per_class_{PER_CLASS}'

import os, shutil, json
def processed_ok(d):
    return os.path.isfile(os.path.join(d, 'meta.json')) and os.path.isfile(os.path.join(d, 'train_points.npy'))

def same_classes(d):
    with open(os.path.join(d, 'meta.json')) as f:
        meta_classes = json.load(f)['classes']
    with open('classes.json') as f:
        return meta_classes == json.load(f)['classes']

if processed_ok(LOCAL_PROCESSED) and same_classes(LOCAL_PROCESSED):
    print('Dataset procesado ya disponible en local.')
elif processed_ok(DRIVE_PROCESSED) and same_classes(DRIVE_PROCESSED):
    print('Restaurando dataset procesado desde Drive...')
    shutil.copytree(DRIVE_PROCESSED, LOCAL_PROCESSED, dirs_exist_ok=True)
else:
    print('Preprocesando (la primera vez tarda: descarga + procesado)...')
    !python preprocess.py --per-class {PER_CLASS} --raw-dir {RAW_CACHE} --out-dir {LOCAL_PROCESSED}
    shutil.copytree(LOCAL_PROCESSED, DRIVE_PROCESSED, dirs_exist_ok=True)
    print('Copia de respaldo en Drive:', DRIVE_PROCESSED)

print(json.load(open(os.path.join(LOCAL_PROCESSED, 'meta.json')))['counts'])

## 7. Prueba rápida (*smoke test*)

Un epoch con 200 dibujos por clase para comprobar que todo funciona de punta a punta antes del
entrenamiento largo. Se guarda en carpetas locales temporales (no en Drive).

In [ ]:
!python train.py --model cnn --data-dir {LOCAL_PROCESSED} --max-per-class 200 --epochs 1 \
    --batch-size 256 --num-workers 2 --run-name smoke_cnn --out-dir /content/smoke/checkpoints \
    --results-dir /content/smoke/results --amp
!python train.py --model bilstm --data-dir {LOCAL_PROCESSED} --max-per-class 200 --epochs 1 \
    --batch-size 256 --num-workers 2 --run-name smoke_bilstm --out-dir /content/smoke/checkpoints \
    --results-dir /content/smoke/results --amp

## 8. Entrenamiento completo de la CNN

* `--amp`: precisión mixta en GPU (más rápido, menos memoria).
* Checkpoints en Drive: `last.pt` cada epoch y `best.pt` con la mejor *top-1* de validación.
* `--resume auto`: si existe `last.pt` del mismo `--run-name`, continúa desde ahí (modelo,
  optimizador, *scheduler*, *scaler*, historial, early stopping y estado aleatorio).
* Programa de *learning rate*: AdamW con calentamiento lineal (1 epoch) y decaimiento coseno.
* *Early stopping* sobre la *top-1* de validación (`--patience`).

In [ ]:
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
NUM_WORKERS = 2   # Colab suele tener 2 CPUs

!python train.py --model cnn --data-dir {LOCAL_PROCESSED} --epochs 30 --batch-size 256 --lr 1e-3 \
    --weight-decay 1e-4 --patience 5 --num-workers {NUM_WORKERS} --image-size 64 --seed 42 \
    --run-name cnn --out-dir {CKPT_DIR} --results-dir {RESULTS_DIR} --amp --resume auto

## 9. Entrenamiento completo de la BiLSTM

Mismo esquema; la entrada son secuencias de hasta `--max-len` puntos. El *gradient clipping*
(norma 1.0 por defecto, `--grad-clip`) estabiliza el entrenamiento de la LSTM.

In [ ]:
!python train.py --model bilstm --data-dir {LOCAL_PROCESSED} --epochs 30 --batch-size 256 --lr 1e-3 \
    --weight-decay 1e-4 --patience 5 --num-workers {NUM_WORKERS} --max-len 250 --seed 42 \
    --run-name bilstm --out-dir {CKPT_DIR} --results-dir {RESULTS_DIR} --amp --resume auto

## 10. Evaluación en *test*

`evaluate.py` calcula *top-1*, *top-3*, pérdida, precisión por clase y matriz de confusión del
mejor checkpoint sobre la partición de test (que no se ha usado ni para entrenar ni para elegir el modelo).

In [ ]:
!python evaluate.py --checkpoint {CKPT_DIR}/cnn/best.pt --data-dir {LOCAL_PROCESSED} --split test --results-dir {RESULTS_DIR}
!python evaluate.py --checkpoint {CKPT_DIR}/bilstm/best.pt --data-dir {LOCAL_PROCESSED} --split test --results-dir {RESULTS_DIR}

## 11. Curvas de entrenamiento y métricas

In [ ]:
import json, os
from IPython.display import Image, display

for run in ('cnn', 'bilstm'):
    run_dir = os.path.join(RESULTS_DIR, run)
    print('=' * 20, run, '=' * 20)
    curves = os.path.join(run_dir, 'curves.png')
    if os.path.isfile(curves):
        display(Image(filename=curves))
    hist_path = os.path.join(run_dir, 'history.json')
    if os.path.isfile(hist_path):
        h = json.load(open(hist_path))
        best = max(range(len(h['epoch'])), key=lambda i: h['val_top1'][i] or 0)
        print(f"epochs: {len(h['epoch'])} | mejor epoch {h['epoch'][best]}: "
              f"val top1 {h['val_top1'][best]:.4f}, val top3 {h['val_top3'][best]:.4f}")
    for name in sorted(os.listdir(run_dir)) if os.path.isdir(run_dir) else []:
        if name.startswith('metrics_') and name.endswith('.json'):
            m = json.load(open(os.path.join(run_dir, name)))
            print(name, {k: v for k, v in m.items() if isinstance(v, (int, float, str))})
        if name.startswith('confusion_') and name.endswith('.png'):
            display(Image(filename=os.path.join(run_dir, name)))

## 12. Copiar los mejores modelos a Drive

Los checkpoints ya están en Drive (`checkpoints/{run}/`), pero se copian también a `best_models/`
con un nombre claro para descargarlos y usarlos en el juego (`checkpoints/cnn/best.pt` y
`checkpoints/bilstm/best.pt` en el proyecto local).

In [ ]:
import shutil, os
for run in ('cnn', 'bilstm'):
    src = f'{CKPT_DIR}/{run}/best.pt'
    if os.path.isfile(src):
        dst = f'{DRIVE_ROOT}/best_models/{run}_best.pt'
        shutil.copy2(src, dst)
        print(f'{src} -> {dst} ({os.path.getsize(dst) / 1e6:.1f} MB)')
    else:
        print('No existe', src)